# Lab 03b - ORM vs PRM (optional)

**Difficulty:** ★★☆→★★★ · **OPTIONAL** · est. build 1–2 days · Scaffold: stubs only.

**Watch/read first**
- Lecture 2, part 2 (Reward models) — https://rlhfbook.com/course/
- Chapter 5, ORM + PRM sections — https://rlhfbook.com/c/05-reward-models.html
- Prerequisite lab: Lab 03 (Bradley-Terry RM) — reuse its pooling discipline.

**Goal:** train two reward models on model-generated solutions — an ORM (binary correctness over GSM8K rollouts) and a PRM ({-1,0,+1} step classifier on a PRM800K slice) — then build the **disagreement case-study table (≥20 solutions)** isolating *correct answer, wrong reasoning* rows.

Do not open the answer key (`rlhf-book/code/reward_models/train_orm.py`, `train_prm.py`) until your version trains.

In [ ]:
# Setup (Colab Tier A / cluster Tier B) -- see labs_curriculum/LABS_SETUP.md
# %pip install "torch>=2.6" "transformers>=4.40" datasets numpy pyyaml
import json, math, random
from pathlib import Path
import numpy as np
import yaml

import starter  # scaffold in this directory
CFG_PATH = Path("configs/03b_orm_vs_prm.yaml")
RUN_DIR = Path("runs/lab03b_orm_vs_prm"); RUN_DIR.mkdir(parents=True, exist_ok=True)
seed = 42; random.seed(seed); np.random.seed(seed)
cfg_d = starter.load_config(str(CFG_PATH)); cfg = starter.OrmPrmLabConfig.from_dict(cfg_d)
print(cfg.model_id, "| amp:", cfg.use_amp, "| pool:", cfg.disagreement_pool_size)

## Part 1 - ORM: outcome supervision

TODO list (all student work):
1. `load_rollout_dataset` + `normalize_rollout_records` -> validate against `ROLLOUT_RECORD_KEYS`
2. `shape_orm_examples` + `orm_collate_fn` (right padding, last-non-pad pooling discipline from Lab 03)
3. `OrmBinaryHead` forward + hand-derived stable `orm_correctness_loss`
4. `run_orm_training_loop` with JSONL metric lines (`starter.append_metric_line`)

 smoke-test at `rollouts_per_prompt_cap: 8` before any long run.

In [ ]:
# TODO(student): implement and run the ORM stage here.
raw = starter.load_rollout_dataset(cfg)                      # raises NotImplementedError
records = starter.normalize_rollout_records(raw)
starter.validate_rollout_records(records)
examples = starter.shape_orm_examples(records, tokenizer=None, cfg=cfg)

## Part 2 - PRM: process supervision

TODO list:
1. `load_prm800k_slice` (respect `prm_slice_rows`; chunk ≤ ~12 steps/problem)
2. `collate_prm_steps`: expand {-1,0,+1} labels to token positions; `-100` ignore_index so 0-labels never silently become negatives
3. `prm_step_loss` (3-way CE, masked mean) + `run_prm_training_loop`

E3 experiment: drop 0-labels vs keep as third class.

In [ ]:
# TODO(student): implement and run the PRM stage here.
prm_records = starter.load_prm800k_slice(cfg)                # raises NotImplementedError
starter.validate_prm_records(prm_records)
batch = starter.collate_prm_steps([{"input_ids": r, "step_bounds": None} for r in []])

## Part 3 - The deliverable: disagreement case-study table

Score ONE shared eval pool under both heads, then:
- verdict rule: sigmoid(logit) >= `eval.orm_positive_threshold` → correct
- PRM flags when min step label == -1
- categories: agree_correct / agree_incorrect / **orm_right_prm_flags_reasoning (the money row)** / prm_right_orm_wrong
- floor: ≥20 rows (config `min_cases_required`); hand-read ≥5 flagged solutions in your write-up.

In [ ]:
# TODO(student): implement dual scoring + table build.
scored = starter.score_solutions_both_models(None, None, [], None)   # NotImplementedError
cases = starter.select_disagreement_cases(scored, cfg.min_cases_required, cfg.orm_positive_threshold)
rows = starter.build_disagreement_table(scored, ground_truth={}, threshold=cfg.orm_positive_threshold)
starter.write_disagreement_table(rows, str(RUN_DIR / "disagreement_table.jsonl"))

## Part 4 - Analysis & write-up

Deliverables per README:
- [ ] category counts plot (how many money rows?)
- [ ] E1 freeze-vs-full comparison; E2 class-balance effect on ORM false positives; E3 ambiguity handling; E4 PRM scarcity
- [ ] ≥5 hand-read case studies quoting the exact poisoned step
- [ ] Compare AFTER training: rlhf-book/code/reward_models/train_orm.py, train_prm.py